# YOLO26 visualisation and second-stage NMS

This notebook keeps the original experiment intact and makes the model output visible while it runs. It writes an annotated video, saves representative annotated frames, and exports the detections retained by an explicit class-aware OpenCV NMS stage.

> Ultralytics already performs its normal confidence filtering and NMS inside `model.predict`. The `NMS_IOU` stage below is an additional, later filter applied to the boxes returned by YOLO. It does not recover predictions removed by YOLO's internal NMS.


In [ ]:
from pathlib import Path
import csv

import cv2
from IPython.display import display, Image
from tqdm.auto import tqdm
from ultralytics import YOLO


In [ ]:
VIDEO_PATH = Path('../data/Neural_net_ninja_video.mp4')
MODEL_NAME = '../yolo26n.pt'
OUTPUT_DIR = Path('runs/yolo26-visualized-nms')
CONFIDENCE = 0.10
IMAGE_SIZE = 640
NMS_IOU = 0.45
MAX_FRAMES = None  # Set to a small integer for a quick test.
PREVIEW_EVERY = 30  # Save and display one annotated frame every N frames.

if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f'Video does not exist: {VIDEO_PATH.resolve()}')
if not 0 < CONFIDENCE <= 1:
    raise ValueError('CONFIDENCE must be greater than 0 and at most 1')
if not 0 < NMS_IOU <= 1:
    raise ValueError('NMS_IOU must be greater than 0 and at most 1')
if PREVIEW_EVERY <= 0:
    raise ValueError('PREVIEW_EVERY must be greater than 0')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_VIDEO = OUTPUT_DIR / f'{VIDEO_PATH.stem}_annotated_nms.mp4'
OUTPUT_CSV = OUTPUT_DIR / f'{VIDEO_PATH.stem}_detections_nms.csv'
PREVIEW_DIR = OUTPUT_DIR / 'previews'
PREVIEW_DIR.mkdir(exist_ok=True)
print(f'Input: {VIDEO_PATH.resolve()}')
print(f'Model: {MODEL_NAME}')
print(f'Output directory: {OUTPUT_DIR.resolve()}')


In [ ]:
def apply_second_stage_nms(result, iou_threshold):
    """Return indices kept by class-aware NMS on a YOLO result."""
    if result.boxes is None or len(result.boxes) == 0:
        return []

    boxes_xyxy = result.boxes.xyxy.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy().tolist()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)
    kept = []
    for class_id in sorted(set(class_ids.tolist())):
        class_indices = [i for i, value in enumerate(class_ids) if value == class_id]
        class_boxes = boxes_xyxy[class_indices]
        xywh_boxes = []
        for x1, y1, x2, y2 in class_boxes:
            xywh_boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
        class_scores = [scores[i] for i in class_indices]
        selected = cv2.dnn.NMSBoxes(xywh_boxes, class_scores, 0.0, iou_threshold)
        kept.extend(class_indices[int(index)] for index in selected)
    return sorted(kept)


def draw_detections(frame, result, indices):
    """Draw selected boxes without relying on Ultralytics' plot helper."""
    annotated = frame.copy()
    boxes = result.boxes
    for index in indices:
        x1, y1, x2, y2 = boxes.xyxy[index].cpu().numpy().astype(int)
        class_id = int(boxes.cls[index])
        confidence = float(boxes.conf[index])
        label = f'{result.names[class_id]} {confidence:.2f}'
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(annotated, label, (x1, max(y1 - 8, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return annotated


In [ ]:
capture = cv2.VideoCapture(str(VIDEO_PATH))
if not capture.isOpened():
    raise RuntimeError(f'Could not open input video: {VIDEO_PATH}')

width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = capture.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
if MAX_FRAMES is not None:
    total_frames = min(total_frames, MAX_FRAMES)
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
if not writer.isOpened():
    capture.release()
    raise RuntimeError(f'Could not create output video: {OUTPUT_VIDEO}')

model = YOLO(MODEL_NAME)
frame_number = 0
before_nms_count = 0
after_nms_count = 0
try:
    with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as csv_file:
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow(['frame', 'class_id', 'class_name', 'confidence', 'x1', 'y1', 'x2', 'y2'])
        with tqdm(total=total_frames, desc='Processing video', unit='frame') as progress:
            while MAX_FRAMES is None or frame_number < MAX_FRAMES:
                ok, frame = capture.read()
                if not ok:
                    break
                result = model.predict(source=frame, conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)[0]
                before_nms_count += len(result.boxes) if result.boxes is not None else 0
                kept_indices = apply_second_stage_nms(result, NMS_IOU)
                after_nms_count += len(kept_indices)
                annotated = draw_detections(frame, result, kept_indices)
                writer.write(annotated)

                if frame_number % PREVIEW_EVERY == 0:
                    preview_path = PREVIEW_DIR / f'frame_{frame_number:06d}.jpg'
                    cv2.imwrite(str(preview_path), annotated)
                    display(Image(filename=str(preview_path)))

                for index in kept_indices:
                    box = result.boxes[index]
                    class_id = int(box.cls)
                    x1, y1, x2, y2 = (float(value) for value in box.xyxy)
                    csv_writer.writerow([frame_number, class_id, result.names[class_id], f'{float(box.conf):.4f}', f'{x1:.2f}', f'{y1:.2f}', f'{x2:.2f}', f'{y2:.2f}'])
                frame_number += 1
                progress.update(1)
finally:
    capture.release()
    writer.release()

print(f'Processed frames: {frame_number}')
print(f'Boxes returned by YOLO: {before_nms_count}')
print(f'Boxes after second-stage NMS: {after_nms_count}')
print(f'Annotated video: {OUTPUT_VIDEO.resolve()}')
print(f'Detections CSV: {OUTPUT_CSV.resolve()}')
print(f'Preview frames: {PREVIEW_DIR.resolve()}')


Open the generated MP4 to inspect every frame. The preview images show the exact boxes written to the video. To make suppression stricter, lower `NMS_IOU`; to keep more overlapping boxes, raise it. For a quick smoke test, set `MAX_FRAMES = 10` before running the processing cell.
